In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
#  Load Dataset
spark = (
    SparkSession.builder
    .appName("ElectricityTheftDetection")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
    )


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/saibhor/Desktop/ELEECTRICITY_THEFT_DETECTION/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/16 23:56:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data_path = "../data/raw/data.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode","PERMISSIVE")
    .csv(data_path)
)

print("Spark version:", spark.version)
print("Rows:", df.count())
print("Columns:", len(df.columns))

df.printSchema()
df.show(5, truncate=False)

Spark version: 4.2.0


Rows: 42372
Columns: 1036
root
 |-- CONS_NO: string (nullable = true)
 |-- FLAG: integer (nullable = true)
 |-- 2014/1/1: double (nullable = true)
 |-- 2014/1/10: double (nullable = true)
 |-- 2014/1/11: double (nullable = true)
 |-- 2014/1/12: double (nullable = true)
 |-- 2014/1/13: double (nullable = true)
 |-- 2014/1/14: double (nullable = true)
 |-- 2014/1/15: double (nullable = true)
 |-- 2014/1/16: double (nullable = true)
 |-- 2014/1/17: double (nullable = true)
 |-- 2014/1/18: double (nullable = true)
 |-- 2014/1/19: double (nullable = true)
 |-- 2014/1/2: double (nullable = true)
 |-- 2014/1/20: double (nullable = true)
 |-- 2014/1/21: double (nullable = true)
 |-- 2014/1/22: double (nullable = true)
 |-- 2014/1/23: double (nullable = true)
 |-- 2014/1/24: double (nullable = true)
 |-- 2014/1/25: double (nullable = true)
 |-- 2014/1/26: double (nullable = true)
 |-- 2014/1/27: double (nullable = true)
 |-- 2014/1/28: double (nullable = true)
 |-- 2014/1/29: double (nullable =

26/10/16 23:59:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------------+----+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+--------+---------+---------+--------+--------+--------+--------+--------+--------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+---------+----------+----------+---------+---------+---------+---------+---------+---------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+---------+----------+---------+---------+---------+---------+---------+---------+-------

In [4]:
print(df.count())
print(len(df.columns))
df.groupBy("FLAG").count().show()

42372
1036


+----+-----+
|FLAG|count|
+----+-----+
|   1| 3615|
|   0|38757|
+----+-----+



In [5]:
print("columns",df.columns[0:])
df.select(df.columns[0], "FLAG").show(5)

columns ['CONS_NO', 'FLAG', '2014/1/1', '2014/1/10', '2014/1/11', '2014/1/12', '2014/1/13', '2014/1/14', '2014/1/15', '2014/1/16', '2014/1/17', '2014/1/18', '2014/1/19', '2014/1/2', '2014/1/20', '2014/1/21', '2014/1/22', '2014/1/23', '2014/1/24', '2014/1/25', '2014/1/26', '2014/1/27', '2014/1/28', '2014/1/29', '2014/1/3', '2014/1/30', '2014/1/31', '2014/1/4', '2014/1/5', '2014/1/6', '2014/1/7', '2014/1/8', '2014/1/9', '2014/10/1', '2014/10/10', '2014/10/11', '2014/10/12', '2014/10/13', '2014/10/14', '2014/10/15', '2014/10/16', '2014/10/17', '2014/10/18', '2014/10/19', '2014/10/2', '2014/10/20', '2014/10/21', '2014/10/22', '2014/10/23', '2014/10/24', '2014/10/25', '2014/10/26', '2014/10/27', '2014/10/28', '2014/10/29', '2014/10/3', '2014/10/30', '2014/10/31', '2014/10/4', '2014/10/5', '2014/10/6', '2014/10/7', '2014/10/8', '2014/10/9', '2014/11/1', '2014/11/10', '2014/11/11', '2014/11/12', '2014/11/13', '2014/11/14', '2014/11/15', '2014/11/16', '2014/11/17', '2014/11/18', '2014/11/19', 

In [6]:

reading_cols = [
    c for c in df.columns
    if c not in ["CONS_NO", "FLAG"]
]

# Convert reading columns to numeric values
df_numeric = df.select(
    F.col("CONS_NO"),
    F.col("FLAG").cast("int").alias("FLAG"),
    *[
        F.col(c).cast("double").alias(c)
        for c in reading_cols
    ]
)

# Create an array of daily readings for each customer
readings_array = F.array(*[F.col(c) for c in reading_cols])

# Ignore null readings when calculating statistics
feature_df = df_numeric.select(
    "CONS_NO",
    "FLAG",
    F.expr(
        "aggregate("
        + "filter(array("
        + ",".join([f"`{c}`" for c in reading_cols])
        + "), x -> x is not null), "
        + "cast(0 as double), "
        + "(acc, x) -> acc + x)"
    ).alias("total_consumption"),
    F.expr(
        "aggregate("
        + "filter(array("
        + ",".join([f"`{c}`" for c in reading_cols])
        + "), x -> x is not null), "
        + "cast(0 as double), "
        + "(acc, x) -> acc + x)"
    ).alias("consumption_sum"),
    F.size(
        F.filter(readings_array, lambda x: x.isNotNull())
    ).alias("valid_readings")
)

feature_df = (
    feature_df
    .withColumn(
        "missing_ratio",
        F.lit(len(reading_cols) - F.col("valid_readings"))
        / F.lit(len(reading_cols))
    )
    .withColumn(
        "mean_consumption",
        F.when(
            F.col("valid_readings") > 0,
            F.col("consumption_sum") / F.col("valid_readings")
        )
    )
)

feature_df.select(
    "CONS_NO", "FLAG", "valid_readings",
    "missing_ratio", "mean_consumption", "total_consumption"
).show(5, truncate=False)


+--------------------------------+----+--------------+---------------------+------------------+-----------------+
|CONS_NO                         |FLAG|valid_readings|missing_ratio        |mean_consumption  |total_consumption|
+--------------------------------+----+--------------+---------------------+------------------+-----------------+
|0387DD8A07E07FDA6271170F86AD9151|1   |591           |0.4284332688588008   |13.89834179357021 |8213.919999999995|
|01D6177B5D4FFE0CABA9EF17DAFC2B84|1   |304           |0.7059961315280464   |0.0               |0.0              |
|4B75AC4F2D8434CFF62DB64D0BB43103|1   |259           |0.7495164410058027   |9.921389961389965 |2569.640000000001|
|B32AC8CC6D5D805AC053557AB05F5343|1   |472           |0.5435203094777563   |14.612055084745764|6896.89          |
|EDFC78B07BA2908B3395C4EB2304665E|1   |1030          |0.0038684719535783366|8.249737864077677 |8497.230000000007|
+--------------------------------+----+--------------+---------------------+------------

In [7]:

# Remove the duplicate column and keep total_consumption
customer_features = feature_df.drop("consumption_sum")

print("Updated columns:", customer_features.columns)


Updated columns: ['CONS_NO', 'FLAG', 'total_consumption', 'valid_readings', 'missing_ratio', 'mean_consumption']


In [8]:

# Define the output directory
output_path = "../data/processed/customer_features"

# Save the customer-level features in Parquet format
(
    customer_features.write
    .mode("overwrite")
    .parquet(output_path)
)

print("Customer features saved successfully!")
print("Saved to:", output_path)


Customer features saved successfully!
Saved to: ../data/processed/customer_features


In [9]:

# Reload the saved Parquet data

features_check = spark.read.parquet(output_path)

print("Columns:", features_check.columns)
print("Number of columns:", len(features_check.columns))
print("Number of rows:", features_check.count())

features_check.show(5, truncate=False)



Columns: ['CONS_NO', 'FLAG', 'total_consumption', 'valid_readings', 'missing_ratio', 'mean_consumption']
Number of columns: 6
Number of rows: 42372
+--------------------------------+----+-----------------+--------------+---------------------+------------------+
|CONS_NO                         |FLAG|total_consumption|valid_readings|missing_ratio        |mean_consumption  |
+--------------------------------+----+-----------------+--------------+---------------------+------------------+
|0387DD8A07E07FDA6271170F86AD9151|1   |8213.919999999995|591           |0.4284332688588008   |13.89834179357021 |
|01D6177B5D4FFE0CABA9EF17DAFC2B84|1   |0.0              |304           |0.7059961315280464   |0.0               |
|4B75AC4F2D8434CFF62DB64D0BB43103|1   |2569.640000000001|259           |0.7495164410058027   |9.921389961389965 |
|B32AC8CC6D5D805AC053557AB05F5343|1   |6896.89          |472           |0.5435203094777563   |14.612055084745764|
|EDFC78B07BA2908B3395C4EB2304665E|1   |8497.2300000000

In [10]:

# Count customers in each target class
class_distribution = (
    features_check
    .groupBy("FLAG")
    .count()
    .orderBy("FLAG")
)

class_distribution.show()

# Calculate class percentages
total_customers = features_check.count()

class_distribution.withColumn(
    "percentage",
    F.round(F.col("count") / total_customers * 100, 2)
).show()


+----+-----+
|FLAG|count|
+----+-----+
|   0|38757|
|   1| 3615|
+----+-----+

+----+-----+----------+
|FLAG|count|percentage|
+----+-----+----------+
|   0|38757|     91.47|
|   1| 3615|      8.53|
+----+-----+----------+



In [11]:

# Summary statistics for the numerical features
features_check.select(
    "valid_readings",
    "missing_ratio",
    "mean_consumption",
    "total_consumption"
).describe().show()


+-------+-----------------+--------------------+-----------------+-----------------+
|summary|   valid_readings|       missing_ratio| mean_consumption|total_consumption|
+-------+-----------------+--------------------+-----------------+-----------------+
|  count|            42372|               42372|            42367|            42372|
|   mean|768.8832247710752| 0.25639920234905755|9.944193224266936|7118.982204797967|
| stddev|298.0590564119112|  0.2882582750598752|82.03726142691521| 34705.2009015358|
|    min|                0|9.671179883945841E-4|              0.0|              0.0|
|    max|             1033|                 1.0|          13601.1|        4080330.0|
+-------+-----------------+--------------------+-----------------+-----------------+



In [12]:

# Compare average feature values for each FLAG class
features_check.groupBy("FLAG").agg(
    F.count("*").alias("customer_count"),
    F.round(F.avg("mean_consumption"), 3).alias("avg_mean_consumption"),
    F.round(F.avg("total_consumption"), 3).alias("avg_total_consumption"),
    F.round(F.avg("missing_ratio"), 3).alias("avg_missing_ratio"),
    F.round(F.avg("valid_readings"), 1).alias("avg_valid_readings")
).orderBy("FLAG").show()


+----+--------------+--------------------+---------------------+-----------------+------------------+
|FLAG|customer_count|avg_mean_consumption|avg_total_consumption|avg_missing_ratio|avg_valid_readings|
+----+--------------+--------------------+---------------------+-----------------+------------------+
|   0|         38757|               7.949|             5963.345|            0.251|             774.5|
|   1|          3615|              31.348|            19508.759|            0.315|             708.6|
+----+--------------+--------------------+---------------------+-----------------+------------------+



In [13]:

from pyspark.sql import functions as F

# 1. Check missing values in each column
missing_summary = features_check.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in features_check.columns
])

missing_summary.show(truncate=False)

# 2. Check duplicate customer IDs
duplicate_ids = (
    features_check
    .groupBy("CONS_NO")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer IDs:", duplicate_ids.count())

# 3. Check invalid values in the engineered features
features_check.select(
    F.sum(
        F.when(F.col("valid_readings") < 0, 1).otherwise(0)
    ).alias("negative_valid_readings"),

    F.sum(
        F.when(
            (F.col("missing_ratio") < 0) |
            (F.col("missing_ratio") > 1),
            1
        ).otherwise(0)
    ).alias("invalid_missing_ratio"),

    F.sum(
        F.when(
            F.col("mean_consumption") < 0, 1
        ).otherwise(0)
    ).alias("negative_mean_consumption"),

    F.sum(
        F.when(
            F.col("total_consumption") < 0, 1
        ).otherwise(0)
    ).alias("negative_total_consumption")
).show()



+-------+----+-----------------+--------------+-------------+----------------+
|CONS_NO|FLAG|total_consumption|valid_readings|missing_ratio|mean_consumption|
+-------+----+-----------------+--------------+-------------+----------------+
|0      |0   |0                |0             |0            |5               |
+-------+----+-----------------+--------------+-------------+----------------+

Duplicate customer IDs: 0
+-----------------------+---------------------+-------------------------+--------------------------+
|negative_valid_readings|invalid_missing_ratio|negative_mean_consumption|negative_total_consumption|
+-----------------------+---------------------+-------------------------+--------------------------+
|                      0|                    0|                        0|                         0|
+-----------------------+---------------------+-------------------------+--------------------------+



In [14]:
model_data = features_check.select(
    "FLAG",
    "valid_readings",
    "missing_ratio",
    "mean_consumption",
    "total_consumption"
)

# stratified split
train_data, test_data = model_data.randomSplit([0.8, 0.2], seed=42)
print("training rows:", train_data.count())
print("testing row: ", test_data.count())
print("trainingclass", train_data.groupBy("FLAG").count().orderBy("FLAG").show())
print("testingclass", test_data.groupBy("FLAG").count().orderBy("FLAG").show())

training rows: 34074
testing row:  8298
+----+-----+
|FLAG|count|
+----+-----+
|   0|31176|
|   1| 2898|
+----+-----+

trainingclass None
+----+-----+
|FLAG|count|
+----+-----+
|   0| 7581|
|   1|  717|
+----+-----+

testingclass None


In [15]:

from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Step 1: Define input features
feature_cols = [
    "valid_readings",
    "missing_ratio",
    "mean_consumption",
    "total_consumption"
]

imputed_cols = [c + "_imputed" for c in feature_cols]

# Step 2: Handle missing numerical values
imputer_standard = Imputer(
    inputCols=feature_cols,
    outputCols=imputed_cols,
    strategy="median"
)

# Step 3: Assemble features into a vector
assembler_standard = VectorAssembler(
    inputCols=imputed_cols,
    outputCol="features"
)

# Step 4: Create standard Logistic Regression
lr_standard = LogisticRegression(
    featuresCol="features",
    labelCol="FLAG",
    maxIter=50,
    regParam=0.01
)

# Step 5: Build the ML pipeline
pipeline_standard = Pipeline(stages=[
    imputer_standard,
    assembler_standard,
    lr_standard
])

# Step 6: Train using the original training dataset
model_standard = pipeline_standard.fit(train_data)

print("Standard Logistic Regression trained successfully!")


Standard Logistic Regression trained successfully!


In [16]:

# Step 1: Calculate class counts from training data only
class_counts = {
    row["FLAG"]: row["count"]
    for row in train_data.groupBy("FLAG").count().collect()
}

count_0 = class_counts.get(0, 0)
count_1 = class_counts.get(1, 0)
total_train = count_0 + count_1

if count_0 == 0 or count_1 == 0:
    raise ValueError("Training data must contain both classes.")

# Step 2: Calculate balanced class weights
weight_0 = total_train / (2.0 * count_0)
weight_1 = total_train / (2.0 * count_1)

print("Weight for FLAG 0:", round(weight_0, 3))
print("Weight for FLAG 1:", round(weight_1, 3))

# Step 3: Add a weight column to each training record
weighted_train = train_data.withColumn(
    "class_weight",
    F.when(F.col("FLAG") == 1, F.lit(weight_1))
     .otherwise(F.lit(weight_0))
)

# Step 4: Define the input features
feature_cols = [
    "valid_readings",
    "missing_ratio",
    "mean_consumption",
    "total_consumption"
]

imputed_cols = [c + "_imputed" for c in feature_cols]

# Step 5: Handle missing values
imputer_weighted = Imputer(
    inputCols=feature_cols,
    outputCols=imputed_cols,
    strategy="median"
)

# Step 6: Assemble features
assembler_weighted = VectorAssembler(
    inputCols=imputed_cols,
    outputCol="features"
)

# Step 7: Create weighted Logistic Regression
lr_weighted = LogisticRegression(
    featuresCol="features",
    labelCol="FLAG",
    weightCol="class_weight",
    maxIter=50,
    regParam=0.01
)

# Step 8: Build the pipeline
pipeline_weighted = Pipeline(stages=[
    imputer_weighted,
    assembler_weighted,
    lr_weighted
])

# Step 9: Train using weighted training data
model_weighted = pipeline_weighted.fit(weighted_train)

print("Weighted Logistic Regression trained successfully!")


Weight for FLAG 0: 0.546
Weight for FLAG 1: 5.879


Weighted Logistic Regression trained successfully!


In [18]:

from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Generate predictions on the same untouched test set
pred_unweighted = model_standard.transform(test_data)
pred_weighted = model_weighted.transform(test_data)

# Evaluator for Precision-Recall AUC
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="FLAG",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

# Evaluator for ROC-AUC
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="FLAG",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

def evaluate_model(predictions, model_name):
    # Confusion matrix counts
    counts = {
        (row["FLAG"], row["prediction"]): row["count"]
        for row in predictions.groupBy("FLAG", "prediction")
                             .count()
                             .collect()
    }

    tn = counts.get((0, 0.0), 0)
    fp = counts.get((0, 1.0), 0)
    fn = counts.get((1, 0.0), 0)
    tp = counts.get((1, 1.0), 0)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    pr_auc = pr_evaluator.evaluate(predictions)
    roc_auc = roc_evaluator.evaluate(predictions)

    print(f"\n--- {model_name} ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print(f"PR-AUC:    {pr_auc:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}")

    print("\nConfusion matrix counts:")
    print(f"True Negatives:  {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    print(f"True Positives:  {tp}")


In [19]:

# Reuse the evaluation function from the previous step
evaluate_model(pred_unweighted, "Standard Logistic Regression")
evaluate_model(pred_weighted, "Weighted Logistic Regression")



--- Standard Logistic Regression ---
Accuracy:  0.9136
Precision: 0.5000
Recall:    0.0042
F1-score:  0.0083
PR-AUC:    0.1783
ROC-AUC:   0.6335

Confusion matrix counts:
True Negatives:  7578
False Positives: 3
False Negatives: 714
True Positives:  3

--- Weighted Logistic Regression ---
Accuracy:  0.6643
Precision: 0.1288
Recall:    0.5007
F1-score:  0.2049
PR-AUC:    0.1814
ROC-AUC:   0.6355

Confusion matrix counts:
True Negatives:  5153
False Positives: 2428
False Negatives: 358
True Positives:  359


In [20]:

from pyspark.ml.classification import RandomForestClassifier

# Numerical features
feature_cols = [
    "valid_readings",
    "missing_ratio",
    "mean_consumption",
    "total_consumption"
]

imputed_cols = [c + "_imputed" for c in feature_cols]

# Preprocessing
rf_imputer = Imputer(
    inputCols=feature_cols,
    outputCols=imputed_cols,
    strategy="median"
)

rf_assembler = VectorAssembler(
    inputCols=imputed_cols,
    outputCol="features"
)

# Weighted Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="FLAG",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=8,
    seed=42
)

rf_pipeline = Pipeline(stages=[
    rf_imputer,
    rf_assembler,
    rf
])

# Train on weighted training data
rf_model = rf_pipeline.fit(weighted_train)

print("Weighted Random Forest trained successfully!")

# Predict on the same test set for initial comparison
pred_rf = rf_model.transform(test_data)

evaluate_model(pred_rf, "Weighted Random Forest")


26/10/17 13:47:59 WARN DAGScheduler: Broadcasting large task binary with size 1669.5 KiB
26/10/17 13:48:03 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB


Weighted Random Forest trained successfully!


26/10/17 13:48:09 WARN DAGScheduler: Broadcasting large task binary with size 1704.5 KiB
26/10/17 13:48:11 WARN DAGScheduler: Broadcasting large task binary with size 1688.0 KiB
26/10/17 13:48:11 WARN DAGScheduler: Broadcasting large task binary with size 1695.8 KiB
26/10/17 13:48:12 WARN DAGScheduler: Broadcasting large task binary with size 1695.8 KiB



--- Weighted Random Forest ---
Accuracy:  0.7087
Precision: 0.1519
Recall:    0.5174
F1-score:  0.2349
PR-AUC:    0.2055
ROC-AUC:   0.6884

Confusion matrix counts:
True Negatives:  5510
False Positives: 2071
False Negatives: 346
True Positives:  371


In [21]:

import pandas as pd

# Get the trained Random Forest from the pipeline
rf_classifier_model = rf_model.stages[-1]

# Extract feature importance scores
importance_scores = rf_classifier_model.featureImportances.toArray()

importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importance_scores
}).sort_values("Importance", ascending=False)

print("Random Forest Feature Importance:")
print(importance_df.to_string(index=False))


Random Forest Feature Importance:
          Feature  Importance
 mean_consumption    0.407488
total_consumption    0.261811
    missing_ratio    0.173833
   valid_readings    0.156868


In [22]:

# Split existing training data into model-training and validation sets
rf_train, validation_data = train_data.randomSplit(
    [0.8, 0.2],
    seed=123
)

print("Random Forest training rows:", rf_train.count())
print("Validation rows:", validation_data.count())

print("\nValidation class distribution:")
validation_data.groupBy("FLAG").count().orderBy("FLAG").show()


Random Forest training rows: 27290
Validation rows: 6784

Validation class distribution:
+----+-----+
|FLAG|count|
+----+-----+
|   0| 6190|
|   1|  594|
+----+-----+



In [23]:

from pyspark.sql import functions as F

class_counts_rf = {
    row["FLAG"]: row["count"]
    for row in rf_train.groupBy("FLAG").count().collect()
}

count_0_rf = class_counts_rf.get(0, 0)
count_1_rf = class_counts_rf.get(1, 0)
total_rf = count_0_rf + count_1_rf

weight_0_rf = total_rf / (2.0 * count_0_rf)
weight_1_rf = total_rf / (2.0 * count_1_rf)

rf_train_weighted = rf_train.withColumn(
    "class_weight",
    F.when(
        F.col("FLAG") == 1,
        F.lit(weight_1_rf)
    ).otherwise(F.lit(weight_0_rf))
)

print("Training class counts:", class_counts_rf)
print(f"Weight for FLAG 0: {weight_0_rf:.4f}")
print(f"Weight for FLAG 1: {weight_1_rf:.4f}")


Training class counts: {1: 2304, 0: 24986}
Weight for FLAG 0: 0.5461
Weight for FLAG 1: 5.9223


In [24]:

from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

feature_cols = [
    "valid_readings",
    "missing_ratio",
    "mean_consumption",
    "total_consumption"
]

imputed_cols = [c + "_imputed" for c in feature_cols]

rf_imputer_val = Imputer(
    inputCols=feature_cols,
    outputCols=imputed_cols,
    strategy="median"
)

rf_assembler_val = VectorAssembler(
    inputCols=imputed_cols,
    outputCol="features"
)

rf_classifier_val = RandomForestClassifier(
    featuresCol="features",
    labelCol="FLAG",
    weightCol="class_weight",
    numTrees=100,
    maxDepth=8,
    seed=42
)

rf_pipeline_val = Pipeline(stages=[
    rf_imputer_val,
    rf_assembler_val,
    rf_classifier_val
])

rf_model_val = rf_pipeline_val.fit(rf_train_weighted)

print("Validation Random Forest trained successfully!")


26/10/17 14:08:10 WARN DAGScheduler: Broadcasting large task binary with size 1650.0 KiB
26/10/17 14:08:12 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB


Validation Random Forest trained successfully!


In [25]:

from pyspark.ml.functions import vector_to_array

validation_predictions = (
    rf_model_val
    .transform(validation_data)
    .withColumn(
        "positive_probability",
        vector_to_array("probability")[1]
    )
    .cache()
)

print("Validation predictions:", validation_predictions.count())

validation_predictions.select(
    "FLAG",
    "positive_probability",
    "prediction"
).show(10, truncate=False)


26/10/17 14:08:39 WARN DAGScheduler: Broadcasting large task binary with size 1788.5 KiB
26/10/17 14:08:41 WARN DAGScheduler: Broadcasting large task binary with size 1796.0 KiB


Validation predictions: 6784


26/10/17 14:08:42 WARN DAGScheduler: Broadcasting large task binary with size 1793.9 KiB


+----+--------------------+----------+
|FLAG|positive_probability|prediction|
+----+--------------------+----------+
|0   |0.5777858784150641  |1.0       |
|0   |0.329146808970815   |0.0       |
|0   |0.5656234004591277  |1.0       |
|0   |0.4742282561016776  |0.0       |
|0   |0.25392818767850256 |0.0       |
|0   |0.5139245709093806  |1.0       |
|0   |0.29978381946821553 |0.0       |
|0   |0.5139245709093806  |1.0       |
|0   |0.25392818767850256 |0.0       |
|0   |0.6690092270805312  |1.0       |
+----+--------------------+----------+
only showing top 10 rows


In [26]:

threshold_results = []

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    scored = validation_predictions.withColumn(
        "predicted_label",
        (F.col("positive_probability") >= threshold).cast("int")
    )

    counts = {
        (row["FLAG"], row["predicted_label"]): row["count"]
        for row in scored.groupBy("FLAG", "predicted_label").count().collect()
    }

    tn = counts.get((0, 0), 0)
    fp = counts.get((0, 1), 0)
    fn = counts.get((1, 0), 0)
    tp = counts.get((1, 1), 0)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_positives": fp,
        "true_positives": tp
    })

threshold_df = pd.DataFrame(threshold_results)

print(
    threshold_df.round(4).to_string(index=False)
)

print("\nBest threshold by validation F1:")
print(
    threshold_df.loc[threshold_df["f1"].idxmax()]
    .to_string()
)


26/10/17 14:08:53 WARN DAGScheduler: Broadcasting large task binary with size 1811.3 KiB
26/10/17 14:08:54 WARN DAGScheduler: Broadcasting large task binary with size 1811.2 KiB
26/10/17 14:08:55 WARN DAGScheduler: Broadcasting large task binary with size 1811.2 KiB
26/10/17 14:08:55 WARN DAGScheduler: Broadcasting large task binary with size 1811.2 KiB
26/10/17 14:08:56 WARN DAGScheduler: Broadcasting large task binary with size 1811.3 KiB
26/10/17 14:08:56 WARN DAGScheduler: Broadcasting large task binary with size 1811.2 KiB
26/10/17 14:08:57 WARN DAGScheduler: Broadcasting large task binary with size 1811.2 KiB
26/10/17 14:08:57 WARN DAGScheduler: Broadcasting large task binary with size 1811.3 KiB
26/10/17 14:08:58 WARN DAGScheduler: Broadcasting large task binary with size 1811.2 KiB
26/10/17 14:08:58 WARN DAGScheduler: Broadcasting large task binary with size 1811.2 KiB
26/10/17 14:08:59 WARN DAGScheduler: Broadcasting large task binary with size 1811.2 KiB


 threshold  precision  recall     f1  false_positives  true_positives
      0.20     0.0952  0.9848 0.1736             5560             585
      0.25     0.0967  0.9680 0.1758             5371             575
      0.30     0.1012  0.9444 0.1828             4984             561
      0.35     0.1073  0.8788 0.1913             4342             522
      0.40     0.1154  0.8047 0.2018             3665             478
      0.45     0.1315  0.7088 0.2218             2781             421
      0.50     0.1668  0.5589 0.2569             1659             332
      0.55     0.1997  0.4562 0.2778             1086             271
      0.60     0.2172  0.3316 0.2625              710             197
      0.65     0.2423  0.2778 0.2588              516             165
      0.70     0.2734  0.1919 0.2255              303             114

Best threshold by validation F1:
threshold             0.550000
precision             0.199705
recall                0.456229
f1                    0.277806
fa

In [27]:

from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Generate test predictions using the validation-trained model
test_predictions = (
    rf_model_val
    .transform(test_data)
    .withColumn(
        "positive_probability",
        vector_to_array("probability")[1]
    )
    .withColumn(
        "predicted_label",
        (F.col("positive_probability") >= 0.55).cast("int")
    )
    .cache()
)

# Calculate confusion matrix
counts = {
    (row["FLAG"], row["predicted_label"]): row["count"]
    for row in (
        test_predictions
        .groupBy("FLAG", "predicted_label")
        .count()
        .collect()
    )
}

tn = counts.get((0, 0), 0)
fp = counts.get((0, 1), 0)
fn = counts.get((1, 0), 0)
tp = counts.get((1, 1), 0)

precision = tp / (tp + fp) if tp + fp else 0
recall = tp / (tp + fn) if tp + fn else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)
f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall else 0
)

# Threshold-independent ranking metrics
pr_auc = BinaryClassificationEvaluator(
    labelCol="FLAG",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
).evaluate(test_predictions)

roc_auc = BinaryClassificationEvaluator(
    labelCol="FLAG",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(test_predictions)

print("--- Weighted Random Forest: Threshold 0.55 ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nConfusion matrix counts:")
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")


26/10/17 14:12:35 WARN DAGScheduler: Broadcasting large task binary with size 1794.5 KiB
26/10/17 14:12:36 WARN DAGScheduler: Broadcasting large task binary with size 1815.7 KiB
26/10/17 14:12:36 WARN DAGScheduler: Broadcasting large task binary with size 1811.6 KiB
26/10/17 14:12:37 WARN DAGScheduler: Broadcasting large task binary with size 1811.6 KiB


--- Weighted Random Forest: Threshold 0.55 ---
Accuracy:  0.7971
Precision: 0.1895
Recall:    0.4114
F1-score:  0.2595
PR-AUC:    0.2054
ROC-AUC:   0.6922

Confusion matrix counts:
True Negatives:  6319
False Positives: 1262
False Negatives: 422
True Positives:  295


In [28]:

import os

model_path = "../models/weighted_random_forest"

os.makedirs("../models", exist_ok=True)

rf_model_val.write().overwrite().save(model_path)

print("Model saved successfully at:", model_path)


26/10/17 14:16:32 WARN TaskSetManager: Stage 353 contains a task of very large size (1051 KiB). The maximum recommended task size is 1000 KiB.


Model saved successfully at: ../models/weighted_random_forest


In [29]:

import os
import json

os.makedirs("../models", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

model_path = "../models/weighted_random_forest"

rf_model_val.write().overwrite().save(model_path)

# Save prediction settings and model metadata
model_config = {
    "project": "Electricity Theft Detection Using Big Data Analytics and Machine Learning",
    "model": "Weighted Random Forest",
    "classification_threshold": 0.55,
    "features": [
        "valid_readings",
        "missing_ratio",
        "mean_consumption",
        "total_consumption"
    ],
    "num_trees": 100,
    "max_depth": 8,
    "label_column": "FLAG"
}

with open("../models/model_config.json", "w") as file:
    json.dump(model_config, file, indent=4)

print("Model and configuration saved successfully.")


26/10/17 14:20:07 WARN TaskSetManager: Stage 366 contains a task of very large size (1051 KiB). The maximum recommended task size is 1000 KiB.


Model and configuration saved successfully.


In [30]:

import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": "Standard Logistic Regression",
        "Threshold": 0.50,
        "Accuracy": 0.9136,
        "Precision": 0.5000,
        "Recall": 0.0042,
        "F1": 0.0083,
        "PR_AUC": 0.1783,
        "ROC_AUC": 0.6335
    },
    {
        "Model": "Weighted Logistic Regression",
        "Threshold": 0.50,
        "Accuracy": 0.6643,
        "Precision": 0.1288,
        "Recall": 0.5007,
        "F1": 0.2049,
        "PR_AUC": 0.1814,
        "ROC_AUC": 0.6355
    },
    {
        "Model": "Weighted Random Forest",
        "Threshold": 0.50,
        "Accuracy": 0.7087,
        "Precision": 0.1519,
        "Recall": 0.5174,
        "F1": 0.2349,
        "PR_AUC": 0.2055,
        "ROC_AUC": 0.6884
    },
    {
        "Model": "Weighted Random Forest",
        "Threshold": 0.55,
        "Accuracy": 0.7971,
        "Precision": 0.1895,
        "Recall": 0.4114,
        "F1": 0.2595,
        "PR_AUC": 0.2054,
        "ROC_AUC": 0.6922
    }
])

comparison.to_csv("../reports/model_comparison.csv", index=False)

print(comparison.round(4).to_string(index=False))
print("\nSaved: ../reports/model_comparison.csv")


                       Model  Threshold  Accuracy  Precision  Recall     F1  PR_AUC  ROC_AUC
Standard Logistic Regression       0.50    0.9136     0.5000  0.0042 0.0083  0.1783   0.6335
Weighted Logistic Regression       0.50    0.6643     0.1288  0.5007 0.2049  0.1814   0.6355
      Weighted Random Forest       0.50    0.7087     0.1519  0.5174 0.2349  0.2055   0.6884
      Weighted Random Forest       0.55    0.7971     0.1895  0.4114 0.2595  0.2054   0.6922

Saved: ../reports/model_comparison.csv


In [31]:

# Save feature importance
importance_df.to_csv(
    "../reports/feature_importance.csv",
    index=False
)

# Save the threshold-0.55 evaluation
final_metrics = {
    "model": "Weighted Random Forest",
    "threshold": 0.55,
    "accuracy": 0.7971,
    "precision": 0.1895,
    "recall": 0.4114,
    "f1_score": 0.2595,
    "pr_auc": 0.2054,
    "roc_auc": 0.6922,
    "true_negatives": 6319,
    "false_positives": 1262,
    "false_negatives": 422,
    "true_positives": 295
}

with open("../reports/final_evaluation.json", "w") as file:
    json.dump(final_metrics, file, indent=4)

print("Feature importance and evaluation saved successfully.")


Feature importance and evaluation saved successfully.


In [32]:

import os

print("Notebook working directory:")
print(os.getcwd())

required_paths = [
    "../models/weighted_random_forest",
    "../models/model_config.json",
    "../reports/model_comparison.csv",
    "../reports/feature_importance.csv",
    "../reports/final_evaluation.json",
]

for path in required_paths:
    status = "FOUND" if os.path.exists(path) else "MISSING"
    print(f"{status}: {path}")


Notebook working directory:
/Users/saibhor/Desktop/ELEECTRICITY_THEFT_DETECTION/notebooks
FOUND: ../models/weighted_random_forest
FOUND: ../models/model_config.json
FOUND: ../reports/model_comparison.csv
FOUND: ../reports/feature_importance.csv
FOUND: ../reports/final_evaluation.json


In [33]:

from pyspark.ml import PipelineModel
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

model_path = "../models/weighted_random_forest"

loaded_model = PipelineModel.load(model_path)

predictions = (
    loaded_model.transform(test_data)
    .withColumn(
        "positive_probability",
        vector_to_array("probability")[1]
    )
    .withColumn(
        "predicted_label",
        (F.col("positive_probability") >= 0.55).cast("int")
    )
)

predictions.select(
    "FLAG",
    "positive_probability",
    "predicted_label"
).show(10, truncate=False)

print("Prediction test completed.")


+----+--------------------+---------------+
|FLAG|positive_probability|predicted_label|
+----+--------------------+---------------+
|0   |0.7268632565400637  |1              |
|0   |0.631988838114257   |1              |
|0   |0.6113962704680769  |1              |
|0   |0.329146808970815   |0              |
|0   |0.6139326064159136  |1              |
|0   |0.6970479124416513  |1              |
|0   |0.29978381946821553 |0              |
|0   |0.5102040638214709  |0              |
|0   |0.4756957573309961  |0              |
|0   |0.6891629008980411  |1              |
+----+--------------------+---------------+
only showing top 10 rows
Prediction test completed.


26/10/17 15:18:58 WARN DAGScheduler: Broadcasting large task binary with size 1862.9 KiB
